# Collecting the EarthPortal data

This notebook downloads everything the audit needs and saves it, untouched, in `data/raw/<RUN_DATE>/`:

- **I.** the catalogue: every public entry of EarthPortal;
- **II.** the portal's own metadata schema: the fields EarthPortal stores for each ontology;
- **III.** the latest submission of every ontology, with all its metadata fields;
- **IV.** the O'FAIRe assessment of every ontology, question by question.

It needs the API key written in the `.env` file at the root of the repository. The other notebooks
only read the files saved here, so they can be run again later on exactly the same data.

In [1]:
# Imports
# 1. Standard library imports
import json
import time

# 2. Third party imports
import pandas as pd
import requests

# 3. Local  imports
from config import *

In [2]:
# what are the imported constants from config.py (capitalised)?
config_vars = sorted(name for name in dir() if name.isupper())
config_vars

['APIKEY',
 'API_URL',
 'AUDITS',
 'CHEATSHEET',
 'CODE_VS_CONFIG',
 'DATA',
 'FOLDER',
 'KNOWN_ISSUES',
 'MANUAL_LINKS',
 'MAPPING',
 'OFAIRE_PROPERTIES',
 'OFAIRE_URL',
 'RAW',
 'REPO',
 'REPORTS',
 'RUN_DATE']

In [3]:
# We check that the API key is there, then prepare a session that sends it in a header

# "assert condition, message" raises an `AssertionError` with that `message` when the `condition` is false
# So without a key the notebook stops here, with a sentence that says what to fix
assert APIKEY, "EARTHPORTAL_APIKEY is empty: write your key in the .env file at the root of the repository"

session = requests.Session() # better than requests.get() because it enables connection reuse to "data.earthportal.eu" and keeps the headers we set once for all requests.
session.headers.update({"Authorization": f"apikey token={APIKEY}", "Accept": "application/json"}) # we ask for JSON, not HTML, and we send the API key in a header
# By default, every object the API returns carries a links block (URLs to related resources) and an @context block (the JSON-LD vocabulary). 
# We do not need either one for the audit, and they make each answer much bigger
QUIET = "display_links=false&display_context=false"     # shorter answers: no links, no JSON-LD context


def get(path, tries=3):
    """
    We ask the EarthPortal API for one path and return its answer as Python objects.

    The path is the part of the address after https://data.earthportal.eu, for example
    "/ontologies/OSO?display_links=false". The API key travels in the session header, never in
    the path. If the network fails or the server answers with an error status (4xx or 5xx),
    we wait 3 seconds and try again, up to `tries` times. After the last failed attempt, the
    original error is raised, so a bad answer is never returned as if it were data.
    """
    for attempt in range(1, tries + 1):
        try:
            answer = session.get(API_URL + path, timeout=60)
            # We send a request and give up if the server does not answer within 60 seconds
            answer.raise_for_status()
            #  raise_for_status() raises an exception if the server returned an error code (4xx := Not Found or 5xx := Server Error),
            # so a bad answer is never saved as if it were data
            return answer.json()
        # RequestException is the parent class of all errors in requests: connection lost, timeout, and the HTTPError raised in step 3
        except requests.RequestException as error:
            # If this was the last attempt, the bare raise sends the original error up to the caller, with its full traceback.
            # Otherwise, it prints what happened, waits 3 seconds (time.sleep(3)), and the loop tries again.
            if attempt == tries:
                raise
            print(f"Retrying {path} after: {error}")
            time.sleep(3)


def save(obj, name):
    """
    We write one API answer, untouched, as a JSON file in the snapshot folder of the day.

    `obj` is the answer returned by get() (a list or a dictionary) and `name` is the file name,
    for example "catalogue.json". The file goes to data/raw/<RUN_DATE>/ and replaces any file of
    the same name already there. Accented characters are kept as they are (UTF-8), so French
    labels stay readable. We print the path of the saved file, relative to the repository root.
    """
    with open(RAW / name, "w", encoding="utf-8") as file:
        # `with... as file`: closes the file automatically, even if an error happens while writing (e.g., disk full, permission denied).
        json.dump(obj, file, ensure_ascii=False, indent=1)
        # indent=1 keeps the file readable line by line, but smaller than the usual indent=4
    print(f"Saved {(RAW / name).relative_to(REPO)}")
    # .relative_to(REPO) prints a short path (earthportal_sa_audit/data/raw/2026-10-06/catalogue.json) instead of the full path.

In [4]:
# Let's try it:
onto = get(f"/ontologies/OSO?{QUIET}")
sorted(onto)          # the field names of one catalogue record

['@id',
 '@type',
 'acronym',
 'administeredBy',
 'flat',
 'group',
 'hasDomain',
 'name',
 'ontologyType',
 'sampleQueries',
 'summaryOnly',
 'viewOf',
 'viewingRestriction',
 'views']

In [5]:
# What about "DOES_NOT_EXIST"?
# get("/ontologies/DOES_NOT_EXIST") # to uncomment...
# Outputs:
# Retrying /ontologies/DOES_NOT_EXIST after: 404 Client Error: Not Found for url: https://data.earthportal.eu/ontologies/DOES_NOT_EXIST
# Retrying /ontologies/DOES_NOT_EXIST after: 404 Client Error: Not Found for url: https://data.earthportal.eu/ontologies/DOES_NOT_EXIST
# HTTPError: 404 Client Error: Not Found for url: https://data.earthportal.eu/ontologies/DOES_NOT_EXIST

# I. The catalogue

In [6]:
# We ask for all public entries, views included
#  get (..) receives the path "/ontologies", everything after "?" is a list of parameters joined by "&":
# /ontologies: The OntoPortal endpoint that lists the ontology records of the portal
# include=all: Return every field of each record. Without it, the API returns only a short default set (acronym, name, ...)
# also_include_views=true: Views are left out by default. We want them, because they are entries of the catalogue too
# QUIET: The two parameters from Block 2 that drop links and @context
catalogue = get(f"/ontologies?include=all&also_include_views=true&{QUIET}")
# we save catalogue: Python list of list of dictionaries, one dictionary per entry.
save(catalogue, "catalogue.json")
print(f"{len(catalogue)} entries in the catalogue")

Saved earthportal_sa_audit/data/raw/2026-10-16/catalogue.json
92 entries in the catalogue


In [7]:
# How does the datacologue look like? we turn the list of dictionaries into a table.. each dict is a row, each key is a column
# If an entry lacks a key, pandas fills that cell with NaN.
df_catalogue = pd.DataFrame(catalogue)
df_catalogue.head()

,id,acronym,administeredBy,viewingRestriction,acl,hasDomain,name,flat,group,ontologyType,summaryOnly,viewOf,views,sampleQueries,doNotUpdate,@id,@type
0,https://data.earthportal.eu/ontologies/CALATHE,CALATHE,[https://data.earthportal.eu/users/vcagdas],public,[],"[https://data.earthportal.eu/categories/EARTH,...",Cadastre and Land Administration Thesaurus,None,[https://data.earthportal.eu/groups/W3C],https://data.earthportal.eu/ontology_types/ONT...,False,None,[],[],None,https://data.earthportal.eu/ontologies/CALATHE,http://data.bioontology.org/metadata/Ontology
1,https://data.earthportal.eu/ontologies/ENVTHES,ENVTHES,[https://data.earthportal.eu/users/ecoportal],public,[],[https://data.earthportal.eu/categories/ENV],Environmental Thesaurus,None,[],https://data.earthportal.eu/ontology_types/ONT...,False,None,[],[],None,https://data.earthportal.eu/ontologies/ENVTHES,http://data.bioontology.org/metadata/Ontology
2,https://data.earthportal.eu/ontologies/N4E,N4E,[https://data.earthportal.eu/users/aurioldegbe...,public,[],[https://data.earthportal.eu/categories/ENV],NFDI4Earth Ontology,None,[https://data.earthportal.eu/groups/NFDI],https://data.earthportal.eu/ontology_types/ONT...,False,None,[],[],None,https://data.earthportal.eu/ontologies/N4E,http://data.bioontology.org/metadata/Ontology
3,https://data.earthportal.eu/ontologies/NVS_C17,NVS_C17,[https://data.earthportal.eu/users/alexk],public,[],"[https://data.earthportal.eu/categories/EARTH,...",ICES Platform Codes,None,[https://data.earthportal.eu/groups/BODC],https://data.earthportal.eu/ontology_types/ONT...,False,None,[],[],None,https://data.earthportal.eu/ontologies/NVS_C17,http://data.bioontology.org/metadata/Ontology
4,https://data.earthportal.eu/ontologies/NVS_C19,NVS_C19,[https://data.earthportal.eu/users/alexk],public,[],"[https://data.earthportal.eu/categories/EARTH,...",SeaVoX salt and fresh water body gazetteer,None,[https://data.earthportal.eu/groups/BODC],https://data.earthportal.eu/ontology_types/ONT...,False,None,[],[],None,https://data.earthportal.eu/ontologies/NVS_C19,http://data.bioontology.org/metadata/Ontology


In [8]:
# Info
df_catalogue.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92 entries, 0 to 91
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   id                  92 non-null     object
 1   acronym             92 non-null     object
 2   administeredBy      92 non-null     object
 3   viewingRestriction  92 non-null     object
 4   acl                 92 non-null     object
 5   hasDomain           92 non-null     object
 6   name                92 non-null     object
 7   flat                2 non-null      object
 8   group               92 non-null     object
 9   ontologyType        92 non-null     object
 10  summaryOnly         77 non-null     object
 11  viewOf              19 non-null     object
 12  views               92 non-null     object
 13  sampleQueries       92 non-null     object
 14  doNotUpdate         0 non-null      object
 15  @id                 92 non-null     object
 16  @type               92 non-n

In [9]:
# A quick inspection of the main fields
df1 = df_catalogue[["acronym", "name", "viewingRestriction", "viewOf"]]
df1.head()

,acronym,name,viewingRestriction,viewOf
0,CALATHE,Cadastre and Land Administration Thesaurus,public,None
1,ENVTHES,Environmental Thesaurus,public,None
2,N4E,NFDI4Earth Ontology,public,None
3,NVS_C17,ICES Platform Codes,public,None
4,NVS_C19,SeaVoX salt and fresh water body gazetteer,public,None


In [10]:
# Are they some non-public entries? (viewingRestriction is not "public")? No ...
df1[df1.viewingRestriction != "public"]

,acronym,name,viewingRestriction,viewOf


In [11]:
# Identify added and removed records between 2026-10-08 and 2026-09-28
# old_file = Path("/u/mdmc/enyanduk/SA_Management/earthportal_sa_audit/data/raw/2026-09-28/catalogue.json")
# old = {onto["acronym"] for onto in json.loads(old_file.read_text(encoding="utf-8"))}
# new = {onto["acronym"]: onto for onto in catalogue}

# print("Added:", sorted(set(new) - old))
# print("Removed:", sorted(old - set(new)))
# for acronym in sorted(set(new) - old):
#     print(acronym, "|", new[acronym]["name"], "|", kind_of(new[acronym]))

# Outputs:
#  Added: ['OSO-SHACL']
#  Removed: []
#  OSO-SHACL | OSO - SHACL (Shapes Constraint Language) | view


#old_kinds = {o["acronym"]: kind_of(o) for o in json.loads(old_file.read_text(encoding="utf-8"))}
#for acronym, onto in new.items():
#    if acronym in old_kinds and old_kinds[acronym] != kind_of(onto):
#        print(acronym, ":", old_kinds[acronym], "->", kind_of(onto), "| viewOf now:", onto.get("viewOf"))

# Outputs:
# OSO-DCAT : artefact -> view | viewOf now: https://data.earthportal.eu/ontologies/OSO
# OSO-VOID : artefact -> view | viewOf now: https://data.earthportal.eu/ontologies/OSO
# Comments:
# OSO-DCAT and OSO-VOID changed from artefact to view, both now declared as views of OSO, and that OSO-SHACL is also a view.
# With those changes, the counts in our catalogue would be 68 artefacts + 19 views + 5 metadata-only = 92.

# II. The portal's own metadata schema

In [12]:
# We ask which metadata fields the portal stores: on each submission (a version of an ontology)
# and on the ontology record itself
schema = {"submission": get(f"/submission_metadata?{QUIET}"),
          "ontology": get(f"/ontology_metadata?{QUIET}")}
save(schema, "metadata_schema.json")
print(f"{len(schema['submission'])} submission fields and {len(schema['ontology'])} ontology fields")
#pd.DataFrame(schema["submission"])[["attribute", "namespace", "category"]].head()

Saved earthportal_sa_audit/data/raw/2026-10-16/metadata_schema.json
110 submission fields and 20 ontology fields


**Notes**

In the blocks above, we downloaded the description of the data: the list of metadata fields that `EarthPortal` itself knows. 
This is the "MOD version used natively by the portal"

It has two end-points : 

| Endpoint               | Describes                                                                                     | Count |
| ---------------------- | --------------------------------------------------------------------------------------------- | ----- |
| `/submission_metadata` | Fields stored on each **submission**, an uploaded version (description, licence, creators,...)| 110   |
| `/ontology_metadata`   | Fields stored on the **ontology record** (acronym, name, administrators, `viewOf`, ...)       | 20    |

In total, 130 fields.

Each element of the list is a dictionary describing one field. The preview keeps three keys:
- `attribute`: the portal's internal name, the one we will meet as a key in `submissions.json` (for example `hasLicense`).
   The first rows (`prefLabelProperty`, `definitionProperty`, `synonymProperty`, `authorProperty`) fall in the category "object description properties". These are a special kind of field. They do not describe the artefact. They tell the portal which property, inside the artefact, holds the labels, definitions, synonyms... of each concept, so they configure how the portal reads the content. That is why the 9 fields of this category have no entry in the guideline.
 
- `namespace`: the vocabulary prefix the field belongs to (`omv`, `dct`, `mod`, ...). It is `None` for portal-internal fields such as `submissionId`, which are not properties of any public vocabulary.
- `category`: the group the field belongs to on the portal's edit form.
- `metadataMappings` is used by notebook 4 to link the portal's names to MOD/guideline names (21 links come from it)

In [13]:
# A quick inspection of the submission metadata fields
df_submission = pd.DataFrame(schema["submission"])
df_submission.head()

,@id,@type,attribute,namespace,label,extracted,metadataMappings,enforce,enforcedValues,category,@context,helpText,description,example
0,http://data.bioontology.org/submission_metadat...,http://data.bioontology.org/metadata/Submissio...,submissionId,None,None,False,None,"[existence, integer, no_list]",None,no,{'@vocab': 'http://data.bioontology.org/metada...,NaN,NaN,NaN
1,http://data.bioontology.org/submission_metadat...,http://data.bioontology.org/metadata/Submissio...,prefLabelProperty,None,Object preferred label property,True,[mod:prefLabelProperty],"[uri, no_list]",{'http://www.w3.org/2004/02/skos/core#prefLabe...,object description properties,{'@vocab': 'http://data.bioontology.org/metada...,Property used to specify objects preferred label.,[MOD: Property used to specify objects preferr...,NaN
2,http://data.bioontology.org/submission_metadat...,http://data.bioontology.org/metadata/Submissio...,definitionProperty,None,Object definition property,True,[mod:definitionProperty],"[uri, no_list]",{'http://www.w3.org/2004/02/skos/core#definiti...,object description properties,{'@vocab': 'http://data.bioontology.org/metada...,Property used to specify objects definition.,[MOD: Property used to specify objects definit...,NaN
3,http://data.bioontology.org/submission_metadat...,http://data.bioontology.org/metadata/Submissio...,synonymProperty,None,Object synonym property,True,[mod:synonymProperty],"[uri, no_list]",{'http://www.w3.org/2004/02/skos/core#altLabel...,object description properties,{'@vocab': 'http://data.bioontology.org/metada...,Property used to specify objects synonyms.,[MOD: Property used to specify objects synonyms.],NaN
4,http://data.bioontology.org/submission_metadat...,http://data.bioontology.org/metadata/Submissio...,authorProperty,None,Object author property,True,[mod:authorProperty],"[uri, no_list]",{'http://purl.org/dc/elements/1.1/creator': 'd...,object description properties,{'@vocab': 'http://data.bioontology.org/metada...,Property used to specify object author.,[MOD: Property used to specify object author.],NaN


In [14]:
# Info
df_submission.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   @id               110 non-null    object
 1   @type             110 non-null    object
 2   attribute         110 non-null    object
 3   namespace         87 non-null     object
 4   label             101 non-null    object
 5   extracted         110 non-null    object
 6   metadataMappings  96 non-null     object
 7   enforce           110 non-null    object
 8   enforcedValues    21 non-null     object
 9   category          110 non-null    object
 10  @context          110 non-null    object
 11  helpText          101 non-null    object
 12  description       100 non-null    object
 13  example           19 non-null     object
dtypes: object(14)
memory usage: 12.2+ KB


In [15]:
# A quick inspection of the submission metadata fields
df2 = df_submission[["attribute", "namespace", "category"]]
df2.head()

,attribute,namespace,category
0,submissionId,None,no
1,prefLabelProperty,None,object description properties
2,definitionProperty,None,object description properties
3,synonymProperty,None,object description properties
4,authorProperty,None,object description properties


In [16]:
# A quick inspection of the Ontology metadata fields
df_ontology = pd.DataFrame(schema["ontology"])
df_ontology.head()

,@id,@type,attribute,namespace,label,extracted,metadataMappings,enforce,enforcedValues,category,@context
0,http://data.bioontology.org/ontology_metadata/...,http://data.bioontology.org/metadata/OntologyM...,acronym,omv,None,False,None,"[unique, existence, no_list]",None,no,{'@vocab': 'http://data.bioontology.org/metada...
1,http://data.bioontology.org/ontology_metadata/...,http://data.bioontology.org/metadata/OntologyM...,name,omv,None,False,None,"[unique, existence, no_list]",None,no,{'@vocab': 'http://data.bioontology.org/metada...
2,http://data.bioontology.org/ontology_metadata/...,http://data.bioontology.org/metadata/OntologyM...,submissions,None,None,False,"[dct:hasVersion, pav:hasCurrentVersion, pav:ha...",[no_list],None,no,{'@vocab': 'http://data.bioontology.org/metada...
3,http://data.bioontology.org/ontology_metadata/...,http://data.bioontology.org/metadata/OntologyM...,projects,None,None,False,None,[no_list],None,no,{'@vocab': 'http://data.bioontology.org/metada...
4,http://data.bioontology.org/ontology_metadata/...,http://data.bioontology.org/metadata/OntologyM...,notes,None,None,False,None,[no_list],None,no,{'@vocab': 'http://data.bioontology.org/metada...


In [17]:
# Info
df_ontology.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   @id               20 non-null     object
 1   @type             20 non-null     object
 2   attribute         20 non-null     object
 3   namespace         4 non-null      object
 4   label             0 non-null      object
 5   extracted         20 non-null     bool  
 6   metadataMappings  3 non-null      object
 7   enforce           20 non-null     object
 8   enforcedValues    0 non-null      object
 9   category          20 non-null     object
 10  @context          20 non-null     object
dtypes: bool(1), object(10)
memory usage: 1.7+ KB


In [18]:
# A quick inspection of the submission metadata fields
df3 = df_ontology[["attribute", "namespace", "category"]]
df3.head()

,attribute,namespace,category
0,acronym,omv,no
1,name,omv,no
2,submissions,None,no
3,projects,None,no
4,notes,None,no


# III. The latest submission of each ontology

In [19]:
# We download the latest submission of each ontology with all its fields (display=all)
submissions, failed = {}, []
for onto in catalogue:
    acronym = onto["acronym"]
    try:
        submissions[acronym] = get(f"/ontologies/{acronym}/latest_submission?display=all&{QUIET}")
    except requests.RequestException:
        submissions[acronym] = {}
        failed.append(acronym)
save(submissions, "submissions.json")
print(f"{len(submissions) - len(failed)} submissions downloaded; failed: {failed or 'none'}")

Saved earthportal_sa_audit/data/raw/2026-10-16/submissions.json
92 submissions downloaded; failed: none


In [20]:
# How does OSO look like?
import pprint
import re

sub = submissions["OSO"]
shown = {key: value for key, value in sub.items() if value not in (None, "", [], {})}
# E-mail addresses are masked, so that no personal address is saved in the notebook output
print(re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+", "***@***", pprint.pformat(shown)))

{'@id': 'https://data.earthportal.eu/ontologies/OSO/submissions/25',
 '@type': 'http://data.bioontology.org/metadata/OntologySubmission',
 'URI': 'https://w3id.org/earthsemantics/OSO',
 'abstract': 'The OSO ontology structures marine observatories into three '
             'hierarchical levels: Regional Facilities, Sites, and Platforms. '
             'It supports FAIR data management, interoperability, and '
             'integration of heterogeneous marine infrastructure information, '
             'particularly within the EMSO ERIC framework.',
 'accrualMethod': ['Curated integration of marine observatory information from '
                   'authoritative research infrastructure and observing '
                   'network sources.'],
 'accrualPeriodicity': 'http://purl.org/cld/freq/continuous',
 'accrualPolicy': 'Active curation and progressive extension. The ontology is '
                  'iteratively improved using FAIR ontology assessment '
                  "results, includin

In [21]:
# A quick inspection of the submission metadata fields
df_last_submission_per_ontology = pd.DataFrame(submissions).T
df_last_submission_per_ontology.head()

,id,ontology,metrics,submissionId,submissionStatus,classType,hasOntologyLanguage,naturalLanguage,prefLabelProperty,definitionProperty,...,hasCreator,publisher,copyrightHolder,hasContributor,translator,endorsedBy,fundedBy,curatedBy,@id,@type
CALATHE,https://data.earthportal.eu/ontologies/CALATHE...,{'administeredBy': ['https://data.earthportal....,{'id': 'https://data.earthportal.eu/ontologies...,1,"[ANNOTATOR, INDEXED, INDEXED_ALL_DATA, INDEXED...",None,SKOS,[],http://www.w3.org/2004/02/skos/core#prefLabel,http://www.w3.org/2004/02/skos/core#definition,...,[{'id': 'https://data.earthportal.eu/Agents/ab...,[],None,[],[],[],[{'id': 'https://data.earthportal.eu/Agents/b3...,[],https://data.earthportal.eu/ontologies/CALATHE...,http://data.bioontology.org/metadata/OntologyS...
ENVTHES,https://data.earthportal.eu/ontologies/ENVTHES...,{'administeredBy': ['https://data.earthportal....,{'id': 'https://data.earthportal.eu/ontologies...,7,"[ANNOTATOR, DIFF, INDEXED, INDEXED_ALL_DATA, I...",None,SKOS,[http://lexvo.org/id/iso639-1/en],http://www.w3.org/2004/02/skos/core#prefLabel,http://www.w3.org/2004/02/skos/core#definition,...,[{'id': 'https://data.earthportal.eu/Agents/32...,[{'id': 'https://data.earthportal.eu/Agents/99...,None,[],[],[],[],[],https://data.earthportal.eu/ontologies/ENVTHES...,http://data.bioontology.org/metadata/OntologyS...
N4E,https://data.earthportal.eu/ontologies/N4E/sub...,{'administeredBy': ['https://data.earthportal....,{'id': 'https://data.earthportal.eu/ontologies...,45,"[ANNOTATOR, DIFF, INDEXED, INDEXED_ALL_DATA, I...",None,OWL,[http://lexvo.org/id/iso639-1/en],http://www.w3.org/2004/02/skos/core#prefLabel,http://www.w3.org/2004/02/skos/core#definition,...,[{'id': 'https://data.earthportal.eu/Agents/5f...,[],None,[{'id': 'https://data.earthportal.eu/Agents/58...,[],[],[],[],https://data.earthportal.eu/ontologies/N4E/sub...,http://data.bioontology.org/metadata/OntologyS...
NVS_C17,https://data.earthportal.eu/ontologies/NVS_C17...,{'administeredBy': ['https://data.earthportal....,{'id': 'https://data.earthportal.eu/ontologies...,72,"[ANNOTATOR, ERROR_DIFF, INDEXED, INDEXED_ALL_D...",None,SKOS,[http://lexvo.org/id/iso639-1/en],http://www.w3.org/2004/02/skos/core#prefLabel,http://www.w3.org/2004/02/skos/core#definition,...,[{'id': 'https://data.earthportal.eu/Agents/d3...,[{'id': 'https://data.earthportal.eu/Agents/ac...,None,[],[],[],[],[],https://data.earthportal.eu/ontologies/NVS_C17...,http://data.bioontology.org/metadata/OntologyS...
NVS_C19,https://data.earthportal.eu/ontologies/NVS_C19...,{'administeredBy': ['https://data.earthportal....,{'id': 'https://data.earthportal.eu/ontologies...,13,"[ANNOTATOR, ERROR_DIFF, INDEXED, INDEXED_ALL_D...",None,SKOS,[http://lexvo.org/id/iso639-1/en],http://www.w3.org/2004/02/skos/core#prefLabel,http://www.w3.org/2004/02/skos/core#definition,...,[],[{'id': 'https://data.earthportal.eu/Agents/ac...,None,[],[],[],[],[],https://data.earthportal.eu/ontologies/NVS_C19...,http://data.bioontology.org/metadata/OntologyS...


# IV. The O'FAIRe assessment

EarthPortal's O'FAIRe service scores each ontology on 61 questions. It needs no API key when it is
called with `portal=earthportal`. We send the acronyms ten at a time to keep each answer small.

In [22]:
acronyms = [onto["acronym"] for onto in catalogue]
ofaire = {}
# range(start, stop, step) gives 0, 10, 20,..., 90. That makes 10 rounds.
for start in range(0, len(acronyms), 10):
    # a slice of 10 acronyms. In the last round, acronyms[90:100] simply returns the 2 remaining ones
    batch = acronyms[start:start + 10]
    # portal=earthportal&ontologies=OSO,ENVTHES,...
    answer = requests.get(OFAIRE_URL, params={"portal": "earthportal", "ontologies": ",".join(batch)}, timeout=300)
    # An HTTP error becomes an exception instead of a silent bad answer.
    answer.raise_for_status()
    # The answer is a dictionary whose key "ontologies" maps each acronym to its result
    ofaire.update(answer.json().get("ontologies", {}))
save(ofaire, "ofaire.json")
missing = [a for a in acronyms if a not in ofaire]
print(f"{len(ofaire)} O'FAIRe results; missing: {missing or 'none'}")

Saved earthportal_sa_audit/data/raw/2026-10-16/ofaire.json
92 O'FAIRe results; missing: none


In [23]:
# A quick inspection:
df_ofaire = pd.DataFrame(ofaire).T
df_ofaire.head()

,Findable,Accessible,Interoperable,Reusable,score,normalizedScore,maxCredits,executionTime
CALATHE,{'F1': {'label': 'Ontologies and ontology meta...,{'A1': {'label': 'Ontologies and ontology meta...,{'I1': {'label': 'Ontologies and ontology meta...,{'R1': {'label': 'Ontologies and ontology meta...,245.0,51,478.0,620
ENVTHES,{'F1': {'label': 'Ontologies and ontology meta...,{'A1': {'label': 'Ontologies and ontology meta...,{'I1': {'label': 'Ontologies and ontology meta...,{'R1': {'label': 'Ontologies and ontology meta...,273.0,57,478.0,525
N4E,{'F1': {'label': 'Ontologies and ontology meta...,{'A1': {'label': 'Ontologies and ontology meta...,{'I1': {'label': 'Ontologies and ontology meta...,{'R1': {'label': 'Ontologies and ontology meta...,254.0,53,478.0,711
NVS_C17,{'F1': {'label': 'Ontologies and ontology meta...,{'A1': {'label': 'Ontologies and ontology meta...,{'I1': {'label': 'Ontologies and ontology meta...,{'R1': {'label': 'Ontologies and ontology meta...,269.75,56,478.0,12976
NVS_C19,{'F1': {'label': 'Ontologies and ontology meta...,{'A1': {'label': 'Ontologies and ontology meta...,{'I1': {'label': 'Ontologies and ontology meta...,{'R1': {'label': 'Ontologies and ontology meta...,264.75,55,478.0,9981


**Conclusion**: the snapshot of the day is saved in `data/raw/<RUN_DATE>/`. The next notebooks read it
from there; to work again on an older snapshot, write its date as `RUN_DATE` in `.env`.